# Template Clustering Signal
load sinyal -> fitur FFT + scale + PCA -> KMeans -> evaluasi -> plot -> insight

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

## Load data

In [ ]:
# data .npz (100rb sinyal x 128 titik): ambil sampel dulu biar ga berat
data = np.load("Signal/SIGNAL_100000_128points.npz")
X_raw = data["x_train"][:5000].astype("float32")
y_true = data["y_train"][:5000]          # label asli, cuma buat ngecek di akhir
class_names = data["class_names"]
time = data["time_seconds"]
print(X_raw.shape)

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(12, 4))
for i, a in enumerate(ax.flat):
    a.plot(time, X_raw[i])
    a.set_title(class_names[y_true[i]], fontsize=8)
plt.tight_layout()
plt.show()

## Preprocessing
Fase sinyal acak, jadi amplitudo mentah susah di-cluster. Pakai FFT (besar tiap frekuensi) + statistik sederhana.

In [ ]:
fft = np.abs(np.fft.rfft(X_raw, axis=1))          # 65 frekuensi (0-64 Hz)
stats = np.column_stack([
    X_raw.mean(axis=1), X_raw.std(axis=1),
    X_raw.max(axis=1), X_raw.min(axis=1),
    np.abs(np.diff(X_raw, axis=1)).mean(axis=1),     # seberapa "kasar" sinyal
])
X = np.hstack([fft, stats])

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(pca.explained_variance_ratio_.sum())

## Tentukan k

In [ ]:
from tqdm.auto import tqdm
ks = range(2, 15)
inertia, sil = [], []
for k in tqdm(ks):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_pca)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_pca, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(ks, inertia, "o-"); ax[0].set_title("Elbow")
ax[1].plot(ks, sil, "o-"); ax[1].set_title("Silhouette")
plt.show()

k dipilih = ... karena ... (isi sendiri dari grafik)

## KMeans + evaluasi

In [ ]:
k = 10   # GANTI
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)

print("Silhouette    :", silhouette_score(X_pca, labels))
print("Davies-Bouldin:", davies_bouldin_score(X_pca, labels))
print(pd.Series(labels).value_counts().sort_index())

## Plot cluster

In [ ]:
p2 = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

plt.figure(figsize=(7, 6))
sc = plt.scatter(p2[:, 0], p2[:, 1], c=labels, cmap="tab10", s=12)
plt.legend(*sc.legend_elements(), title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title("KMeans (PCA 2D)")
plt.show()

## Insight tiap cluster

In [ ]:
# 3 contoh sinyal per cluster (rata-rata ga guna karena fase acak)
fig, ax = plt.subplots(k, 3, figsize=(10, 1.6 * k), sharex=True)
for c in range(k):
    idx = np.where(labels == c)[0][:3]
    for j, i in enumerate(idx):
        ax[c, j].plot(time, X_raw[i], lw=0.8)
    ax[c, 0].set_ylabel(f"C{c} n={np.sum(labels == c)}", fontsize=8)
plt.tight_layout()
plt.show()

# spektrum rata-rata tiap cluster
freqs = np.fft.rfftfreq(128, d=1/128)
plt.figure(figsize=(9, 4))
for c in range(k):
    plt.plot(freqs, fft[labels == c].mean(axis=0), label=f"C{c}")
plt.xlabel("Frekuensi (Hz)"); plt.legend(ncol=2, fontsize=8)
plt.title("Rata-rata spektrum FFT per cluster")
plt.show()

In [ ]:
# bandingin ke label asli kalau ada
pd.crosstab(labels, pd.Series(class_names[y_true], name="kelas"))

## Kesimpulan
- Evaluasi: silhouette = ..., DBI = ...
- Cluster 0: ...
- Cluster 1: ...
- Cluster yang campur: ...